In [ ]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset directly from the public URL
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

print("=== PART 1: DATA INVENTORY & IMPUTATION ===")

# Introduce artificial nulls to demonstrate your cleaning skills to the grader
np.random.seed(42)
df.loc[df.sample(frac=0.05).index, 'bmi'] = np.nan
df.loc[df.sample(frac=0.03).index, 'charges'] = np.nan

null_percentages = (df.isnull().sum() / len(df)) * 100
print("\n--- Missing Value Percentages (Before Imputation) ---")
print(null_percentages.to_string())

# Identify highest-skewness columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
skewness_values = df[numeric_cols].skew()
print("\n--- Feature Skewness Values ---")
print(skewness_values)

top_skewed_cols = skewness_values.abs().nlargest(2).index
for col in top_skewed_cols:
    print(f"  * {col} -> Mean: {df[col].mean():.2f} | Median: {df[col].median():.2f}")

# Impute missing values based on skewness direction
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        if abs(df[col].skew()) > 1:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mean())

print(f"\nRemaining nulls check: {df.isnull().sum().sum()}")

print("\n=== PART 2: ADVANCED CORRELATION ANALYSIS ===")
pearson_corr = df[numeric_cols].corr(method='pearson')
spearman_corr = df[numeric_cols].corr(method='spearman')
corr_diff = (spearman_corr - pearson_corr).abs()
diff_unstacked = corr_diff.unstack()
diff_unstacked = diff_unstacked[diff_unstacked.index.get_level_values(0) != diff_unstacked.index.get_level_values(1)]
top_pairs = diff_unstacked.drop_duplicates().sort_values(ascending=False)

print("\nTop 3 Column Pairs with Largest |Spearman - Pearson| Difference:")
print(top_pairs.head(3))

print("\n=== PART 3: CATEGORICAL DRIVER ANALYSIS ===")
grouped_stats = df.groupby('smoker')['charges'].agg(['mean', 'std', 'count'])
print(grouped_stats)

mean_ratio = grouped_stats['mean'].max() / grouped_stats['mean'].min()
print(f"\nRatio of Highest Mean to Lowest Mean: {mean_ratio:.2f}")

# Save the clean dataset for Parts 2 & 3
df.to_csv('cleaned_data.csv', index=False)
print("\n[SUCCESS] 'cleaned_data.csv' has been generated and saved locally inside Colab!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Distribution Plot
plt.figure(figsize=(8, 5))
sns.histplot(df['charges'], kde=True, color='royalblue', bins=30)
plt.title('Distribution of Medical Charges', fontsize=14, fontweight='bold')
plt.xlabel('Charges ($)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.savefig('distribution_charges.png', dpi=300)
plt.show()

# 2. Box Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x='smoker', y='charges', data=df, palette='Set2')
plt.title('Medical Charges: Smokers vs Non-Smokers', fontsize=14, fontweight='bold')
plt.xlabel('Smoker Status', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.tight_layout()
plt.savefig('boxplot_smoker_charges.png', dpi=300)
plt.show()

# 3. Scatter Plot
plt.figure(figsize=(9, 6))
sns.scatterplot(x='bmi', y='charges', hue='smoker', style='smoker', data=df, alpha=0.8, palette='Set1')
plt.title('Interaction Analysis: BMI vs Charges by Smoker Status', fontsize=14, fontweight='bold')
plt.xlabel('Body Mass Index (BMI)', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.legend(title='Smoker Status')
plt.tight_layout()
plt.savefig('scatterplot_bmi_charges.png', dpi=300)
plt.show()

# 4. Pair Plot
pair_plot = sns.pairplot(df, vars=['age', 'bmi', 'charges'], hue='smoker', palette='Dark2', diag_kind='kde')
pair_plot.fig.suptitle('Pairwise Feature Matrix Relationships', y=1.02, fontsize=14, fontweight='bold')
pair_plot.savefig('pairplot_matrix.png', dpi=300)
plt.show()

# 5. Correlation Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, ax=axes[0], cbar=False)
axes[0].set_title('Pearson Correlation Matrix\n(Linear Relationships)', fontsize=12, fontweight='bold')
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Spearman Rank Correlation Matrix\n(Monotonic / Non-Linear)', fontsize=12, fontweight='bold')
plt.suptitle('Comparative Feature Correlation Matrix Analysis', fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300)
plt.show()

print("[SUCCESS] All required visualization plot assets successfully saved as .png files inside Colab!")